 UCI 477 Real Estate Valuation Pipeline

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42
TEST_SIZE = 1 / 3
CV_FOLDS = 5
CV_SCORING = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}

In [ ]:
ds = fetch_ucirepo(id=477)
print("features:", ds.data.features.columns.tolist())
print("targets:", ds.data.targets.columns.tolist())
print("ids:", ds.data.ids.columns.tolist())
X = ds.data.features.copy()
y = ds.data.targets.iloc[:, 0]

features: ['X1 transaction date', 'X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude']
targets: ['Y house price of unit area']
ids: ['No']


In [ ]:
df = X.copy()
df["Y"] = y.values
id_cols = [c for c in df.columns if c.lower() in ("no", "id")]
if id_cols:
    df = df.drop(columns=id_cols)
print("shape:", df.shape)
print("missing:", df.isna().sum().sum())
print("duplicates:", df.duplicated().sum())
print("describe:\n", df.describe())

shape: (414, 7)
missing: 0
duplicates: 0
describe:
        X1 transaction date  X2 house age  \
count           414.000000    414.000000   
mean           2013.148971     17.712560   
std               0.281967     11.392485   
min            2012.667000      0.000000   
25%            2012.917000      9.025000   
50%            2013.167000     16.100000   
75%            2013.417000     28.150000   
max            2013.583000     43.800000   

       X3 distance to the nearest MRT station  \
count                              414.000000   
mean                              1083.885689   
std                               1262.109595   
min                                 23.382840   
25%                                289.324800   
50%                                492.231300   
75%                               1454.279000   
max                               6488.021000   

       X4 number of convenience stores  X5 latitude  X6 longitude           Y  
count                       4

In [ ]:
feature_cols = [c for c in df.columns if c != "Y"]
X_all, y_all = df[feature_cols], df["Y"]
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print("train:", X_train.shape[0], "test:", X_test.shape[0])

train: 276 test: 138


In [ ]:
train_df = X_train.copy()
train_df["Y"] = y_train
print(train_df.describe())
print("\ncorrelation:\n", train_df.corr())
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, train_df.columns):
    ax.hist(train_df[col], bins=20, edgecolor="black")
    ax.set_title(col, fontsize=8)
plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=100)
plt.close()
corr = train_df.corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(corr.columns)), corr.columns, fontsize=7)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=100)
plt.close()

       X1 transaction date  X2 house age  \
count           276.000000    276.000000   
mean           2013.169399     17.402174   
std               0.280516     11.379599   
min            2012.667000      0.000000   
25%            2012.917000      8.475000   
50%            2013.208500     16.100000   
75%            2013.417000     28.250000   
max            2013.583000     42.700000   

       X3 distance to the nearest MRT station  \
count                              276.000000   
mean                              1051.099834   
std                               1241.491872   
min                                 49.661050   
25%                                286.786750   
50%                                492.231300   
75%                               1408.531750   
max                               6306.153000   

       X4 number of convenience stores  X5 latitude  X6 longitude           Y  
count                       276.000000   276.000000    276.000000  276.000000  
m

In [ ]:
models = {
    "baseline_mean": DummyRegressor(strategy="mean"),
    "linear_regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "random_forest": RandomForestRegressor(random_state=RANDOM_STATE),
    "gradient_boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "svr": Pipeline([("scaler", StandardScaler()), ("model", SVR())]),
}
cv_rows = []
for name, model in models.items():
    s = cross_validate(model, X_train, y_train, cv=CV_FOLDS, scoring=CV_SCORING, n_jobs=-1)
    cv_rows.append({"model": name, "cv_mae": -s["test_mae"].mean(), "cv_rmse": -s["test_rmse"].mean(), "cv_r2": s["test_r2"].mean()})
cv_results = pd.DataFrame(cv_rows).sort_values("cv_r2", ascending=False)
print(cv_results.to_string(index=False))

            model    cv_mae   cv_rmse     cv_r2
    random_forest  5.336437  8.132161  0.651280
gradient_boosting  5.426449  8.272212  0.638129
linear_regression  6.475778  9.039074  0.573108
              svr  6.504574  9.541401  0.521942
    baseline_mean 10.907462 14.002089 -0.038548


In [ ]:
best_name = cv_results.iloc[0]["model"]
best_model = models[best_name]
best_model.fit(X_train, y_train)
test_pred = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
test_r2 = r2_score(y_test, test_pred)
print(f"best model: {best_name}")
print("test:", {"mae": test_mae, "rmse": test_rmse, "r2": test_r2})

best model: random_forest
test: {'mae': 4.466647041062804, 'rmse': 6.573061287389175, 'r2': 0.7289564227872101}
